In [ ]:
# Late Binding behavior of closures in loops.
def create_tasks():
    tasks = []
    for i in range(3):
        tasks.append(lambda: i)
    return tasks

for t in create_tasks():
    print(t()) # Prints 2, 2, 2 (Not 0, 1, 2)
    
# second example with fix ------------------------------

# The Bug
funcs = [lambda: i for i in range(3)]
print([f() for f in funcs])  # Expected [0, 1, 2], Got [2, 2, 2]

# The Fix
funcs = [lambda i=i: i for i in range(3)]
func_results = [f() for f in funcs]
print(func_results)  # Correctly prints [0, 1, 2]

2
2
2
[2, 2, 2]
[0, 1, 2]


In [ ]:
import time
import functools
from typing import Callable, Any

def rate_limit(seconds: float):
    """
    Factory that returns a decorator. 
    'seconds' is part of the closure's environment.
    """
    def decorator(func: Callable):
        # Enclosed state: This variable lives as long as the 
        # decorated function exists.
        last_called = 0.0

        @functools.wraps(func)  # CRITICAL: Preserves __name__, __doc__, etc.
        def wrapper(*args, **kwargs) -> Any:
            nonlocal last_called  # Allows us to modify the outer scope variable
            
            elapsed = time.time() - last_called
            if elapsed < seconds:
                raise RuntimeError(f"Rate limit exceeded. Wait {seconds - elapsed:.2f}s")
            
            result = func(*args, **kwargs)
            last_called = time.time()
            return result
        
        return wrapper
    return decorator

# --- Usage in a Domain Service ---

@rate_limit(seconds=2)
def request_api_data(user_id: int):
    """Fetches sensitive data."""
    return {"user": user_id, "status": "success"}

# Execution
print(request_api_data(101)) # Works
# print(request_api_data(101)) # Raises RuntimeError if called within 2s